# FlameOn Pipeline 3 — Colab GPU Batch Transcription

Runs the same Pipeline 3 processing (silence detection, trimming, compression, loudness normalization, Whisper transcription, timestamp remapping) as the local `pipeline3_transcribe.py` but on a Colab T4 GPU for ~10-15x speedup.

## Runtime requirements
**Runtime → Change runtime type → GPU (T4)**

## Input / Output
- **Input:** a folder of audio/video files on Google Drive
- **Output:** one `<case_id>_transcript.json` per input file, matching the `p3_to_p4_transcript` schema

## Expected speed
- 15-min interview MP3: ~1 min
- 1-hour BWC MP4: ~4-6 min
- Batch of 10 files (~3 hours of audio): ~15-20 min total

## 1. Verify GPU is available

In [ ]:
!nvidia-smi

## 2. Install dependencies
faster-whisper for GPU transcription (CTranslate2 backend, native CUDA)

In [ ]:
!pip install -q faster-whisper
!apt-get install -y ffmpeg > /dev/null

## 3. Mount Google Drive
The expected layout is:
```
/content/drive/MyDrive/FlameOn/
  case_0409-18/           # input audio/video files
    *.mp3
    *.mp4
  transcripts/            # output JSONs written here
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
INPUT_DIR = '/content/drive/MyDrive/FlameOn/case_0409-18'
OUTPUT_DIR = '/content/drive/MyDrive/FlameOn/transcripts'
os.makedirs(OUTPUT_DIR, exist_ok=True)

files = [f for f in sorted(os.listdir(INPUT_DIR)) if f.lower().endswith(('.mp3', '.mp4', '.mov', '.wav', '.m4a'))]
print(f'Found {len(files)} files to process:')
for f in files:
    sz = os.path.getsize(os.path.join(INPUT_DIR, f)) / 1024 / 1024
    print(f'  {sz:>6.1f} MB  {f}')

## 4. Pipeline 3 helper functions

These are the same functions as `pipeline3_transcribe.py`, inlined here for the notebook. Silence detection + trimming + loudness normalization + offset map remapping — everything Pipeline 4 needs to reference original video timestamps.

In [ ]:
import subprocess
import re
import json
import tempfile
import shutil
import time

TARGET_SR = 16000
SILENCE_THRESHOLD_DB = -40
MIN_SILENCE_SEC = 2.0
LOUDNESS_TARGET = -16.0


def _ffmpeg_to_wav(src, dst):
    cmd = ['ffmpeg', '-y', '-i', src, '-ar', str(TARGET_SR), '-ac', '1', '-vn', dst]
    subprocess.run(cmd, capture_output=True, check=True)


def analyze(wav):
    cmd = ['ffprobe', '-v', 'error', '-show_entries', 'stream=sample_rate,channels',
           '-show_entries', 'format=duration,size', '-of', 'json', wav]
    r = subprocess.run(cmd, capture_output=True, text=True, check=True)
    data = json.loads(r.stdout)
    return {
        'duration_sec': float(data['format']['duration']),
        'size_bytes': int(data['format'].get('size', 0)),
    }


def detect_silence(wav, thresh=SILENCE_THRESHOLD_DB, min_sec=MIN_SILENCE_SEC):
    cmd = ['ffmpeg', '-i', wav, '-af', f'silencedetect=noise={thresh}dB:d={min_sec}', '-f', 'null', '-']
    r = subprocess.run(cmd, capture_output=True, text=True)
    starts = [float(m.group(1)) for m in re.finditer(r'silence_start:\s*([\d.]+)', r.stderr)]
    ends = [float(m.group(1)) for m in re.finditer(r'silence_end:\s*([\d.]+)', r.stderr)]
    return [{'original_start_sec': round(s, 3),
             'original_end_sec': round(e, 3),
             'duration_sec': round(e - s, 3)}
            for s, e in zip(starts, ends)]


def build_offset_map(silence, total_dur):
    keep = []
    cursor = 0.0
    for sp in sorted(silence, key=lambda x: x['original_start_sec']):
        if sp['original_start_sec'] > cursor:
            keep.append((cursor, sp['original_start_sec']))
        cursor = max(cursor, sp['original_end_sec'])
    if cursor < total_dur:
        keep.append((cursor, total_dur))
    offsets = []
    tc = 0.0
    for s, e in keep:
        offsets.append((round(tc, 3), round(s, 3)))
        tc += (e - s)
    return offsets, keep, round(tc, 3)


def trimmed_to_original(t_sec, offset_map, keep):
    result = 0.0
    for i, (tc, oc) in enumerate(offset_map):
        if t_sec >= tc:
            delta = t_sec - tc
            result = oc + delta
            if i < len(keep) and result > keep[i][1]:
                result = keep[i][1]
        else:
            break
    return round(result, 3)


def trim_silence(wav, keep_segs, out):
    if not keep_segs:
        shutil.copy(wav, out); return
    filters = [f'[0:a]atrim=start={s}:end={e},asetpts=PTS-STARTPTS[a{i}]' for i, (s, e) in enumerate(keep_segs)]
    concat = ''.join(f'[a{i}]' for i in range(len(keep_segs)))
    fc = ';'.join(filters) + f';{concat}concat=n={len(keep_segs)}:v=0:a=1[out]'
    subprocess.run(['ffmpeg', '-y', '-i', wav, '-filter_complex', fc, '-map', '[out]',
                    '-ar', str(TARGET_SR), '-ac', '1', out], capture_output=True, check=True)


def compress_and_normalize(wav, out, target=LOUDNESS_TARGET):
    # compand then loudnorm in a single pipeline
    af = (f'compand=attacks=0.3:decays=0.8:points=-80/-80|-45/-15|-27/-9|0/-7|20/-7:soft-knee=6:gain=0,'
          f'loudnorm=I={target}:TP=-1.5:LRA=11')
    subprocess.run(['ffmpeg', '-y', '-i', wav, '-af', af, '-ar', str(TARGET_SR), '-ac', '1', out],
                   capture_output=True, check=True)

## 5. Load Whisper model (GPU)

`large-v3` on T4 with float16 is the sweet spot: maximum accuracy + fits in T4's 16GB VRAM comfortably.

In [ ]:
from faster_whisper import WhisperModel

MODEL_NAME = 'large-v3'  # or 'medium' for faster but slightly less accurate
print(f'Loading {MODEL_NAME} model on GPU (float16)...')
t = time.time()
model = WhisperModel(MODEL_NAME, device='cuda', compute_type='float16')
print(f'  Loaded in {time.time()-t:.1f}s')

## 6. Process each file through the full pipeline

In [ ]:
def process_file(input_path, case_id, evidence_type, source_url=None):
    print(f'\n{"="*70}\nProcessing: {os.path.basename(input_path)}\n{"="*70}')
    t_start = time.time()
    work = tempfile.mkdtemp(prefix=f'p3_{case_id}_')

    try:
        # 1. Convert to 16kHz mono WAV
        print('  [1/7] Converting to WAV...')
        original_wav = os.path.join(work, 'original.wav')
        _ffmpeg_to_wav(input_path, original_wav)

        # 2. Analyze
        stats = analyze(original_wav)
        dur = stats['duration_sec']
        print(f'  [2/7] Duration: {dur:.1f}s ({dur/60:.1f} min)')

        # 3. Detect silence
        silence = detect_silence(original_wav)
        print(f'  [3/7] Detected {len(silence)} silence segments ({sum(s["duration_sec"] for s in silence):.1f}s total)')

        # 4. Build offset map + trim
        offset_map, keep, trimmed_dur = build_offset_map(silence, dur)
        print(f'  [4/7] Kept {len(keep)} segments, trimmed duration: {trimmed_dur:.1f}s')
        trimmed_wav = os.path.join(work, 'trimmed.wav')
        trim_silence(original_wav, keep, trimmed_wav)

        # 5. Compress + normalize in one pass
        print('  [5/7] Dynamic range compression + loudness normalization...')
        norm_wav = os.path.join(work, 'normalized.wav')
        compress_and_normalize(trimmed_wav, norm_wav)

        # 6. Transcribe on GPU
        print(f'  [6/7] Transcribing with {MODEL_NAME} on GPU...')
        t_w = time.time()
        segments, info = model.transcribe(norm_wav, language='en', beam_size=5,
                                          word_timestamps=True, vad_filter=False)
        segments = list(segments)
        print(f'    Got {len(segments)} segments in {time.time()-t_w:.1f}s ({info.language}, {info.language_probability:.2f})')

        # 7. Remap timestamps to original timeline
        print('  [7/7] Remapping timestamps trimmed→original...')
        transcript = []
        for seg in segments:
            words = getattr(seg, 'words', None) or []
            confs = [getattr(w, 'probability', None) for w in words if getattr(w, 'probability', None) is not None]
            conf = round(sum(confs) / len(confs), 3) if confs else None
            transcript.append({
                'start_sec': trimmed_to_original(round(seg.start, 3), offset_map, keep),
                'end_sec': trimmed_to_original(round(seg.end, 3), offset_map, keep),
                'text': seg.text.strip(),
                'confidence': conf,
            })

        output = {
            'case_id': case_id,
            'source_evidence_type': evidence_type,
            'source_url': source_url or os.path.basename(input_path),
            'transcript': transcript,
            'silence_map': silence,
            'original_duration_sec': round(dur, 3),
            'processed_duration_sec': trimmed_dur,
            'speaker_count': None,
            'processing_metadata': {
                'whisper_model': MODEL_NAME,
                'silence_threshold_db': SILENCE_THRESHOLD_DB,
                'min_silence_duration_sec': MIN_SILENCE_SEC,
                'compression_applied': True,
                'loudness_target_lufs': LOUDNESS_TARGET,
                'total_processing_sec': round(time.time() - t_start, 1),
                'device': 'cuda_float16',
            },
        }

        out_path = os.path.join(OUTPUT_DIR, f'{case_id}_transcript.json')
        with open(out_path, 'w', encoding='utf-8') as f:
            json.dump(output, f, indent=2, ensure_ascii=False)

        print(f'  ✓ Saved: {out_path}')
        print(f'  Total: {time.time() - t_start:.1f}s for {dur/60:.1f} min audio ({(dur / (time.time() - t_start)):.1f}x realtime)')
        return output

    finally:
        shutil.rmtree(work, ignore_errors=True)

## 7. Run the batch

Case 0409-18 = 4 audio/video files (2 BWC MP4 + 2 DPA interview MP3). The PDF is processed separately in step 8.

In [ ]:
# Map files to case_id and evidence_type
# Edit these to match your specific files
BATCH = [
    # (filename_pattern, case_id, evidence_type)
    ('BWC of Officer Sherry',    'sfdpa_0409-18_bwc_sherry',    'bodycam'),
    ('BWC of Sergeant Bradford', 'sfdpa_0409-18_bwc_bradford',  'bodycam'),
    ('Interview of Officer Sherry',    'sfdpa_0409-18_int_sherry',    'interrogation'),
    ('Interview of Sergeant Bradford', 'sfdpa_0409-18_int_bradford',  'interrogation'),
]

results = []
batch_start = time.time()
for pattern, case_id, evidence_type in BATCH:
    matches = [f for f in files if pattern in f]
    if not matches:
        print(f'\n[SKIP] No file matching {pattern!r}')
        continue
    input_path = os.path.join(INPUT_DIR, matches[0])
    result = process_file(input_path, case_id, evidence_type)
    results.append(result)

print(f'\n{"="*70}')
print(f'BATCH COMPLETE — {len(results)} files processed in {(time.time()-batch_start)/60:.1f} min')
print(f'{"="*70}')
for r in results:
    print(f'  {r["case_id"]}: {r["original_duration_sec"]/60:.1f} min → {len(r["transcript"])} segments')

## 8. (Optional) Extract case narrative from production PDF
The 422MB production PDF contains the incident report, witness statements, use-of-force analysis, and exhibits. Extract the first ~50 pages where the narrative summary usually lives.

In [ ]:
!pip install -q pypdf
from pypdf import PdfReader

pdf_path = os.path.join(INPUT_DIR, 'Production - 0409-18.pdf')
if os.path.exists(pdf_path):
    reader = PdfReader(pdf_path)
    print(f'PDF has {len(reader.pages)} pages')
    # Extract first 30 pages into a text file
    text_parts = []
    for i in range(min(30, len(reader.pages))):
        text = reader.pages[i].extract_text() or ''
        text_parts.append(f'=== PAGE {i+1} ===\n{text}\n')
    out = os.path.join(OUTPUT_DIR, 'sfdpa_0409-18_case_narrative.txt')
    with open(out, 'w', encoding='utf-8') as f:
        f.write('\n'.join(text_parts))
    print(f'Saved first 30 pages to {out}')
else:
    print(f'PDF not found at {pdf_path}')